# Gold Layer Aggregation Pipeline - Generic & Metadata-Driven

Generic notebook to combine, join, and aggregate Silver tables into Gold business marts based on config.

In [0]:
# Notebook Parameters
dbutils.widgets.text("config_dir", "", "Config Directory (Optional - Auto-resolved if blank)")
dbutils.widgets.text("business_domain", "sales", "Business Domain")
dbutils.widgets.text("target_table", "customer_summary", "Target Gold Table")
# Stage choices come from the central stage registry (configs/environments/env_settings.json keys)
import json as _json
from pathlib import Path as _Path

def _stage_registry_choices():
    fallback = ["dev", "test", "staging", "prod"]
    bases = []
    try:
        _cfg = dbutils.widgets.get("config_dir").strip()
        if _cfg:
            bases += [_Path(_cfg), _Path(_cfg).parent]
    except Exception:
        pass
    try:
        _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        _here = _Path("/Workspace" + _nb).parent
        bases += [_here, *_here.parents]
    except Exception:
        pass
    bases += [_Path.cwd(), *_Path.cwd().parents]
    for _base in bases:
        for _reg in (_base / "configs" / "environments" / "env_settings.json",
                     _base / "environments" / "env_settings.json"):
            try:
                if _reg.is_file():
                    _stages = list(_json.loads(_reg.read_text(encoding="utf-8")).keys())
                    if _stages:
                        return _stages
            except Exception:
                continue
    return fallback

_STAGE_CHOICES = _stage_registry_choices()
dbutils.widgets.dropdown("environment", "dev" if "dev" in _STAGE_CHOICES else _STAGE_CHOICES[0], _STAGE_CHOICES, "Environment / Stage")
dbutils.widgets.text("client_id", "", "Client ID / Tenant (Optional)")
dbutils.widgets.dropdown("create_views", "false", ["false", "true"], "Create Views (Optional)")

# Resolve config_dir and repo_root generically
import os
import sys
from pathlib import Path

raw_config_dir = ""
try:
    raw_config_dir = dbutils.widgets.get("config_dir").strip()
except Exception:
    raw_config_dir = ""

if raw_config_dir:
    config_dir = raw_config_dir
    repo_root = str(Path(config_dir).parent)
else:
    try:
        # Dynamically discover the notebook's workspace/repo location
        nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        repo_root = str(Path(nb_path).parent.parent)
        config_dir = str(Path(repo_root) / "configs")
    except Exception:
        cwd = Path(os.getcwd())
        if (cwd / "configs").exists():
            repo_root = str(cwd)
            config_dir = str(cwd / "configs")
        elif (cwd.parent / "configs").exists():
            repo_root = str(cwd.parent)
            config_dir = str(cwd.parent / "configs")
        else:
            repo_root = str(cwd)
            config_dir = "configs"

# Workspace file access from cluster compute requires the /Workspace prefix
if not config_dir.startswith("/Workspace") and os.path.isdir("/Workspace" + config_dir):
    config_dir = "/Workspace" + config_dir
if not repo_root.startswith("/Workspace") and os.path.isdir("/Workspace" + repo_root):
    repo_root = "/Workspace" + repo_root

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

business_domain = dbutils.widgets.get("business_domain")
target_table = dbutils.widgets.get("target_table")
try:
    _stage = dbutils.widgets.get("stage").strip()
except Exception:
    _stage = ""
environment = _stage or dbutils.widgets.get("environment")
client_id = dbutils.widgets.get("client_id").strip() or None
create_views = dbutils.widgets.get("create_views").lower() == "true"

print(f"Executing Gold Aggregation for {target_table} (client: {client_id or 'global'}, {business_domain} domain, {environment} environment)")


In [0]:
# Imports & Path setup
import sys

from lakeforge import (
    ConfigParser,
    GoldAggregator,
    create_trust_engine
)

print("✅ Framework libraries loaded")

In [0]:
# Load Configurations
env_config = ConfigParser.get_environment_config(config_dir, environment)
catalog = env_config["catalog"]
silver_schema = env_config["silver_schema"]
gold_schema = env_config["gold_schema"]

gold_config = ConfigParser.get_gold_aggregation_config(config_dir, target_table, environment, client_id=client_id)
print(f"Parsed Gold Config for: {target_table}")
print(gold_config)


In [0]:
# Load Silver source tables dynamically
silver_dfs = {}
for alias, table_path in gold_config["source_tables"].items():
    # Extract basename, e.g., 'silver.customers_clean' -> 'customers_clean'
    table_basename = table_path.split(".")[-1]
    full_silver_path = f"{catalog}.{silver_schema}.{table_basename}"
    
    print(f"Loading source: {alias} -> {full_silver_path}")
    silver_dfs[alias] = spark.table(full_silver_path)

print(f"✅ Loaded {len(silver_dfs)} source Silver dataframes")

In [0]:
# Execute aggregations dynamically
result_df = GoldAggregator.aggregate(spark, silver_dfs, gold_config)
print(f"✅ Aggregation complete. Columns: {result_df.columns}")

In [0]:
# Write / Merge to Gold layer
merge_strat = gold_config.get("merge_strategy", "overwrite")
business_keys = gold_config.get("business_key", [])

full_gold_table = f"{catalog}.{gold_schema}.{target_table}"

if merge_strat in ["upsert", "merge"] and business_keys:
    print(f"ℹ️ Performing merge/upsert on Gold table matching keys {business_keys}...")
    from lakeforge.silver.merge_engine import SilverMergeEngine
    merge_stats = SilverMergeEngine.merge(
        spark=spark,
        df=result_df,
        target_table=full_gold_table,
        merge_keys=business_keys,
        mode="merge"
    )
    print(f"✅ Merge complete: {merge_stats}")
elif merge_strat == "append":
    print("ℹ️ Performing append on Gold table...")
    from lakeforge.silver.merge_engine import SilverMergeEngine
    merge_stats = SilverMergeEngine.merge(
        spark=spark,
        df=result_df,
        target_table=full_gold_table,
        merge_keys=[],
        mode="append"
    )
    print(f"✅ Append complete: {merge_stats}")
else:
    print("ℹ️ Performing complete overwrite on Gold table...")
    result_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(full_gold_table)
    print(f"✅ Overwrite complete")

In [0]:
# Compute and Log Trust Score
trust_engine = create_trust_engine(spark)

# Compare first source table row count with target aggregated row count for reconciliation
first_source = list(silver_dfs.values())[0]
trust_validations = [
    {
        "type": "row_count",
        "params": {
            "source_df": first_source,
            "target_df": result_df,
            "tolerance_percent": 100.0  # Gold aggregations naturally compress row counts, so tolerance is high
        }
    }
]

trust_results = trust_engine.run_trust_validations(trust_validations)
trust_score = trust_engine.calculate_trust_score(
    table_name=full_gold_table,
    dq_results=trust_results,
    pipeline_stage="gold"
)

print(f"🎯 Pipeline Trust Score: {trust_score['overall_score']:.1f}%")
print(f"🎯 Trust Level: {trust_score['trust_level']}")

In [0]:
# Create Views if explicitly enabled (otherwise run VIEWS_Creation_Pipeline on-demand)
if create_views:
    try:
        print("Checking for configured views to create...")
        from lakeforge.views.view_manager import create_view_manager
        from lakeforge.metadata.config_parser import load_view_configs
        env_config = ConfigParser.get_environment_config(config_dir, environment)
        view_configs = load_view_configs(config_dir, environment, client_id=client_id)
        target_views = [vc for vc in view_configs if vc.schema == env_config["gold_schema"]]
        if target_views:
            view_manager = create_view_manager(spark, env_config)
            view_manager.create_views(target_views)
            print(f"Created {len(target_views)} views successfully")
        else:
            print("No views configured for gold schema")
    except Exception as e:
        print(f"Error creating views: {str(e)}")
else:
    print("View creation skipped (decoupled; trigger VIEWS_Creation_Pipeline on demand or set create_views=true)")
